<a href="https://colab.research.google.com/github/vbhandeo-bits/NLP_ASSIGNMENT_2/blob/main/Assignment2_NLP_PS21.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CS F429 / IS F429: Natural Language Processing
## Assignment 2: Grammatical Error Correction (GEC) as Sequence-to-Sequence Translation

This notebook contains a complete assignment solution for grammatical error correction using a transformer-based sequence-to-sequence model. It includes:

- Environment setup and package installation
- Dataset loading and preprocessing for `agentlans/grammar-correction`
- Fine-tuning `t5-small` with early stopping and validation
- Over-correction handling for safer model generation
- Evaluation using BERTScore, WER, and CER
- A local Gradio interface for interactive inference

<div style="padding: 15px; border: 1px solid #4CAF50; border-radius: 5px; background-color: #f9f9f9;">
<b>Team Details:</b><br>
<ul>
<li><b> Student 1</b>
<li><b> BITS ID: 2025AB05033</b>
<li><b> Name: VAIBHAV BHANDEO</b>
<li><b> Email: 2025AB05033@wilp.bits-pilani.ac.in</b>
<li><b> Student 2</b>
<li><b> BITS ID: 2025aa05448</b>
<li><b> Name: VAIBHAVI VISHWANATH BADIGER</b>
<li><b> Email: 2025aa05448@wilp.bits-pilani.ac.in</b>
<li><b> Group No: 93</b>
<li><b> Subject: NLP</b>
<li><b> Date: 09-August-2026</b>
<li><b> Student 3</b>
<li><b> BITS ID: 2025aa05606</b>
<li><b> Name: V Raj Kumar</b>
<li><b> Email: 2025aa05606@wilp.bits-pilani.ac.in</b>
<li><b> Group No: 93</b>
<li><b> Subject: NLP</b>
<li><b> Date: 09-August-2026</b>
<li><b> Student 4</b>
<li><b> BITS ID: 2025AA05610</b>
<li><b> Name: VSSGG Rahul Mugada</b>
<li><b> Email: 2025AA05610@wilp.bits-pilani.ac.in</b>
<li><b> Group No: 93</b>
<li><b> Subject: NLP</b>
<li><b> Date: 09-August-2026</b>
<li><b> Student 5</b>
<li><b> BITS ID: 2025aa05165</b>
<li><b> Name: Rohit Vadje</b>
<li><b> Email: 2025aa05165@wilp.bits-pilani.ac.in</b>
<li><b> Group No: 93</b>
<li><b> Subject: NLP</b>
<li><b> Date: 09-August-2026</b>
  <li><b>Environment:</b> <b>BITS CSIS Labs / CUDA GPU Cluster</b></li>
</ul>
</div>


### 1. Environment Setup & Hardware Acceleration Check
This section ensures the notebook has all required dependencies installed and verifies whether CUDA acceleration is available. The package installation is performed inside the notebook kernel so that imports work immediately and reproducibly.

We install:
- `transformers` for model and tokenizer APIs
- `datasets` for dataset loading and preprocessing
- `evaluate` for metric computation
- `bert-score` and `jiwer` for evaluation metrics
- `gradio` for a local interactive interface
- `torch` and `accelerate` for training support

In [1]:
# Install required dependencies
%pip install -q transformers datasets evaluate bert-score jiwer gradio torch accelerate

import torch
import numpy as np
import pandas as pd
import gradio as gr
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback
)
import evaluate
from jiwer import wer, cer

# Validate BITS CSIS Lab GPU Execution
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[INFO] Active Execution Device: {device}")
if device == "cuda":
    print(f"[INFO] GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"[INFO] Available Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("[WARNING] CUDA is not available; training will run on CPU and may be slower.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 42.6 MB/s eta 0:00:00
[INFO] Active Execution Device: cuda
[INFO] GPU Model: Tesla T4
[INFO] Available Memory: 15.64 GB


---
### Phase 1: Data Preparation & Preprocessing
In this section, we load the grammar correction dataset and transform raw text into model-ready token sequences. We use a task prefix so that the T5 model learns the correct inference objective, and we apply consistent padding and masking for stable batched training.

Key preprocessing steps:
- add a task prompt to each source sentence
- tokenize inputs and targets with padding/truncation
- replace padding token IDs with `-100` in labels so the loss ignores padding positions

In [2]:
# 1. Load Dataset
dataset_name = "agentlans/grammar-correction"
raw_dataset = load_dataset(dataset_name)
print("Dataset Structure:", raw_dataset)

# 2. Tokenizer Initialization
MODEL_CHECKPOINT = "t5-small"
PREFIX = "grammar correction: "
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

# 3. Preprocessing Routine
def preprocess_function(examples):
    inputs = [PREFIX + text for text in examples["input"]]
    targets = examples["output"]

    # Tokenize the source text with fixed length padding/truncation.
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length")

    # Tokenize the target corrected text separately.
    labels = tokenizer(text_target=targets, max_length=128, truncation=True, padding="max_length")

    # Replace padding IDs in the target sequence with -100 so they are ignored by the loss.
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Map data splits
tokenized_dataset = raw_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=raw_dataset["train"].column_names
)

if "validation" not in tokenized_dataset:
    split_data = tokenized_dataset["train"].train_test_split(test_size=0.1, seed=42)
    train_dataset = split_data["train"]
    val_dataset = split_data["test"]
else:
    train_dataset = tokenized_dataset["train"]
    val_dataset = tokenized_dataset["validation"]

README.md:   0%|          | 0.00/3.14k [00:00<?, ?B/s]

train.jsonl.zst: reconstructing file:   0%|          |  0.00B / 6.91MB            

train.jsonl.zst: downloading bytes:           |  0.00B            

validation.jsonl.zst: reconstructing file:   0%|          |  0.00B / 1.77MB            

validation.jsonl.zst: downloading bytes:           |  0.00B            

...

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

---
### Phase 2: Model Fine-Tuning & Training
This section fine-tunes `t5-small` for grammatical error correction using the Hugging Face `Seq2SeqTrainer`. We configure evaluation at each epoch and apply early stopping based on validation loss to avoid overfitting and save the best model checkpoint.

Training highlights:

- model: `t5-small` for efficient lab-scale training
- `predict_with_generate=True` so evaluation uses real generation outputs
- early stopping: stops if validation loss does not improve for 2 epochs

In [3]:
# Load base Seq2Seq model
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT)

# Training Configs
training_args = Seq2SeqTrainingArguments(
    output_dir="./gec_t5_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=5,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=100,
    report_to="none"
)

# Data collator and Early Stopping (2 epochs patience)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)
early_stopping = EarlyStoppingCallback(early_stopping_patience=2, early_stopping_threshold=0.001)

# Initialize Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    callbacks=[early_stopping]
)

# Run Fine-Tuning
trainer.train()

# Save best weights after training completes
model.save_pretrained("./gec_t5_best")
tokenizer.save_pretrained("./gec_t5_best")

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss
1,0.783942,0.721942
2,0.736856,0.710152
3,0.664863,0.711411
4,0.642388,0.709687


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./gec_t5_best/tokenizer_config.json', './gec_t5_best/tokenizer.json')

---
### Phase 3: Handling Over-Correction
This section demonstrates how generation hyperparameters can help avoid unnecessary rewrites when the source sentence is already grammatical. We show both unconstrained generation and mitigated decoding to compare behavior.

Key ideas:
- use beam search and repetition penalties to reduce over-correction
- apply a task-guided prefix and model evaluation for safer output

In [4]:
def generate_corrected_text(input_text, mitigate_overcorrection=True):
    formatted_input = PREFIX + input_text
    inputs = tokenizer(formatted_input, return_tensors="pt", max_length=128, truncation=True).to(device)

    model.eval()
    with torch.no_grad():
        if mitigate_overcorrection:
            # Tuned generation to mitigate stylistic over-correction
            outputs = model.generate(
                **inputs,
                max_length=128,
                num_beams=4,
                length_penalty=0.8,
                repetition_penalty=1.1,
                no_repeat_ngram_size=3,
                early_stopping=True
            )
        else:
            # Default unconstrained greedy decode
            outputs = model.generate(**inputs, max_length=128, num_beams=1)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Testing samples sensitive to over-correction
samples = [
    "I shall head to the library post haste.",
    "This solution works fine for our current scenario."
]

print("=== OVER-CORRECTION TEST DEMONSTRATION ===")
for text in samples:
    raw_out = generate_corrected_text(text, mitigate_overcorrection=False)
    mitigated_out = generate_corrected_text(text, mitigate_overcorrection=True)
    print(f"Original Sentence:    '{text}'")
    print(f"Unconstrained Model:  '{raw_out}'")
    print(f"Mitigated Decoding:   '{mitigated_out}'")
    print("-" * 60)

=== OVER-CORRECTION TEST DEMONSTRATION ===
Original Sentence:    'I shall head to the library post haste.'
Unconstrained Model:  'I shall head to the library post grab.'
Mitigated Decoding:   'I shall head to the library post grab.'
------------------------------------------------------------
Original Sentence:    'This solution works fine for our current scenario.'
Unconstrained Model:  'This solution works fine for our current scenario.'
Mitigated Decoding:   'This solution works fine for our current scenario.'
------------------------------------------------------------


---
### Phase 4: Evaluation Metrics & Error Diagnostic Analysis
This evaluation section computes metrics to compare model outputs with reference corrections. It highlights both success cases and challenging failure cases with diagnostics to explain why errors persist.

Metrics used:
- **BERTScore** for semantic similarity and meaning preservation
- **WER** for word-level edit distance
- **CER** for character-level error rate

The result table makes it easy to see which corrections are complete, which are partial, and where the model still struggles.

In [5]:
bertscore_metric = evaluate.load("bertscore")

# Evaluation samples (5 Success, 3 Failures)
test_sources = [
    # Success Cases
    "He go to school yesterday morning.",
    "She don't likes apples for breakfast.",
    "They is playing football outside in rain.",
    "I have been living here since five years.",
    "Where you are going right now?",
    # Failure Cases
    "The model's performence were sub-optimum due to hardware limits.",
    "GNNs handles graph data very well.",
    "Irregardless of the outcome, we must proceed."
]

test_targets = [
    "He went to school yesterday morning.",
    "She doesn't like apples for breakfast.",
    "They are playing football outside in the rain.",
    "I have been living here for five years.",
    "Where are you going right now?",
    "The model's performance was sub-optimal due to hardware limits.",
    "GNNs handle graph data very well.",
    "Regardless of the outcome, we must proceed."
]

# Compute Predictions
predictions = [generate_corrected_text(s, mitigate_overcorrection=True) for s in test_sources]

# Compute Metrics
bert_scores = bertscore_metric.compute(predictions=predictions, references=test_targets, lang="en")["f1"]

results = []
for idx in range(len(test_sources)):
    src = test_sources[idx]
    ref = test_targets[idx]
    pred = predictions[idx]

    results.append({
        "Type": "Success" if idx < 5 else "Failure",
        "Input Source": src,
        "Target Reference": ref,
        "Model Prediction": pred,
        "BERTScore F1": round(bert_scores[idx], 4),
        "WER": round(wer(ref, pred), 4),
        "CER": round(cer(ref, pred), 4)
    })

df_results = pd.DataFrame(results)
display(df_results)

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


,Type,Input Source,Target Reference,Model Prediction,BERTScore F1,WER,CER
0,Success,He go to school yesterday morning.,He went to school yesterday morning.,He went to school yesterday morning.,1.0000,0.0000,0.0000
1,Success,She don't likes apples for breakfast.,She doesn't like apples for breakfast.,She doesn't like apples for breakfast.,1.0000,0.0000,0.0000
2,Success,They is playing football outside in rain.,They are playing football outside in the rain.,They are playing football outside in the rain.,1.0000,0.0000,0.0000
3,Success,I have been living here since five years.,I have been living here for five years.,I have been living here for five years.,1.0000,0.0000,0.0000
4,Success,Where you are going right now?,Where are you going right now?,Where are you going right now?,1.0000,0.0000,0.0000
5,Failure,The model's performence were sub-optimum due t...,The model's performance was sub-optimal due to...,The model's performance was sub-optimum due to...,0.9945,0.1111,0.0317
6,Failure,GNNs handles graph data very well.,GNNs handle graph data very well.,GNNs handles graph data very well.,0.9952,0.1667,0.0303
7,Failure,"Irregardless of the outcome, we must proceed.","Regardless of the outcome, we must proceed.","Irregardless of the outcome, we must proceed.",0.9713,0.1429,0.0698


#### Failure Diagnostics Breakdown
1. **Out-of-Vocabulary / Technical Terms (`GNNs`, `sub-optimum`):** T5's SentencePiece subword tokenizer decomposes domain-specific acronyms into sub-units, causing the decoder to miss verb-subject agreements for non-standard terminology.
2. **Non-Standard Colloquial Invariances (`Irregardless`):** Rare non-standard terms have low frequency in pre-training distributions, causing standard generation beams to pass them through unchanged.
3. **Long-Distance Dependencies:** Complex multi-clause sentences occasionally lose agreement context across long decoding step horizons in lightweight backbones like `t5-small`.

---
### Phase 5: Web Application Interface (Gradio Deployment)
This final section builds a local Gradio app for interactive sentence correction. The interface lets users type noisy text and see real-time grammar corrections generated by the trained model.

In [6]:
def predict_gec_interface(user_input):
    if not user_input or not user_input.strip():
        return "Please enter a valid sentence."
    return generate_corrected_text(user_input, mitigate_overcorrection=True)

# Construct Web App
interface = gr.Interface(
    fn=predict_gec_interface,
    inputs=gr.Textbox(lines=3, placeholder="Enter ungrammatical or noisy sentence...", label="Input Noisy Text"),
    outputs=gr.Textbox(label="Corrected Output"),
    title="Grammatical Error Correction (T5 Sequence-to-Sequence)",
    description="Fine-tuned lightweight transformer model translating noisy text into fluent English.",
    examples=[
        ["He go to school yesterday."],
        ["She dont likes apples."],
        ["Where you are going?"]
    ]
)

# Launch Interface locally
interface.launch(share=False, inline=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>